# Risk Prediction Model for Patient Progress Monitoring
## Objective 3: ML Pipeline for At-Risk Patient Detection

**Goal:** Predict patients at risk of early return visits within 30 days
**Target:** `return_within_30d` binary classification


## 1. Setup & Data Load


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# LightGBM (optional but recommended)
try:
    import lightgbm as lgb
    HAS_LGBM = True
    print("✅ LightGBM available")
except ImportError:
    print("⚠️ LightGBM not installed. Install with: pip install lightgbm")
    HAS_LGBM = False

# SHAP (optional, for explainability)
try:
    import shap
    HAS_SHAP = True
    print("✅ SHAP available for feature importance")
except ImportError:
    print("⚠️ SHAP not installed (optional). Install with: pip install shap")
    HAS_SHAP = False

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("\n📦 Using models: RandomForest, GradientBoosting, LightGBM")
print("="*60)


In [ ]:
# Preprocessing pipeline: load -> rebuild labels -> utilization -> features -> clean

def preprocess_raw_data(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path, parse_dates=['visit_date']).copy()
    
    # Sort for timeline ops
    df = df.sort_values(['patient_id', 'visit_date']).reset_index(drop=True)
    
    # Rebuild return labels from next visit
    df['next_visit_date'] = df.groupby('patient_id')['visit_date'].shift(-1)
    df['days_to_next'] = (df['next_visit_date'] - df['visit_date']).dt.days
    df['return_within_14d'] = ((df['days_to_next'] >= 0) & (df['days_to_next'] <= 14)).astype(int)
    df['return_within_30d'] = ((df['days_to_next'] >= 0) & (df['days_to_next'] <= 30)).astype(int)
    
    # Rolling prior utilization
    def rolling_count_within_days(x_dates, window):
        out = []
        for i, d in enumerate(x_dates):
            lo = d - pd.Timedelta(days=window)
            out.append(((x_dates[:i] >= lo) & (x_dates[:i] < d)).sum())
        return pd.Series(out, index=x_dates.index)
    
    df['visits_30d']  = df.groupby('patient_id')['visit_date'].transform(lambda s: rolling_count_within_days(s, 30))
    df['visits_90d']  = df.groupby('patient_id')['visit_date'].transform(lambda s: rolling_count_within_days(s, 90))
    df['visits_365d'] = df.groupby('patient_id')['visit_date'].transform(lambda s: rolling_count_within_days(s, 365))
    
    # Last gap
    df['prev_visit_date'] = df.groupby('patient_id')['visit_date'].shift(1)
    df['days_since_last_visit'] = (df['visit_date'] - df['prev_visit_date']).dt.days.fillna(9999)
    
    # Replace prior_visits_90d with true value
    if 'prior_visits_90d' in df.columns:
        df['prior_visits_90d'] = df['visits_90d']
    else:
        df['prior_visits_90d'] = df['visits_90d']
    
    # Temporal/context features
    df['dow'] = df['visit_date'].dt.dayofweek
    df['month'] = df['visit_date'].dt.month
    df['is_rainy'] = df['month'].isin([6,7,8,9]).astype(int)
    df['visit_month_period'] = df['visit_date'].dt.to_period('M').astype(str)
    
    # Provider congestion proxy per month
    if 'provider_id' in df.columns:
        load = df.groupby(['provider_id','visit_month_period']).size().rename('provider_month_load').reset_index()
        df = df.merge(load, on=['provider_id','visit_month_period'], how='left')
    else:
        df['provider_month_load'] = df.groupby('visit_month_period').visit_month_period.transform('count')
    
    # Chronic indicators from diagnosis
    if 'diagnosis_group' in df.columns:
        df['is_htn'] = (df['diagnosis_group'] == 'Hypertension').astype(int)
        df['is_dm'] = (df['diagnosis_group'] == 'Diabetes Mellitus').astype(int)
        df['is_asthma'] = (df['diagnosis_group'] == 'Asthma').astype(int)
    else:
        df['is_htn'] = 0; df['is_dm'] = 0; df['is_asthma'] = 0
    
    # Vital risk flags (with coercion)
    for c in ['systolic_bp','diastolic_bp','heart_rate','bmi']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    df['bp_high'] = ((df.get('systolic_bp',0) >= 140) | (df.get('diastolic_bp',0) >= 90)).astype(int)
    df['tachy'] = (df.get('heart_rate',0) >= 100).astype(int)
    df['obese'] = (df.get('bmi',0) >= 30).astype(int)
    
    # Drop leakage columns if present
    leakage_cols = ['next_followup_date','followup_recommended','progress_status']
    for c in leakage_cols:
        if c in df.columns:
            df = df.drop(columns=[c])
    
    return df


In [ ]:
# Run preprocessing and load dataset
print("Running preprocessing pipeline...")
df_visits = preprocess_raw_data('mmc_opd_visits_2023_2025.csv')
print(f"Rows: {len(df_visits):,}  |  True 30d return rate: {df_visits['return_within_30d'].mean():.2%}")
print(df_visits[['patient_id','visit_date','return_within_30d','visits_90d','days_since_last_visit']].head())


In [ ]:
## 1.2 Recompute True Return Labels
# ====================================
# Fix: The original labels might be synthetic/flat
# Rebuild 'return_within_30d' from actual patient visit timelines

print("Step 1: Rebuilding return labels from visit history...")
df_visits = df_visits.sort_values(['patient_id', 'visit_date']).copy()

# Calculate time to next visit
df_visits['next_visit_date'] = df_visits.groupby('patient_id')['visit_date'].shift(-1)
df_visits['days_to_next'] = (df_visits['next_visit_date'] - df_visits['visit_date']).dt.days

# Rebuild return labels (0-14 days, 0-30 days)
df_visits['return_within_14d_true'] = ((df_visits['days_to_next'] >= 0) & (df_visits['days_to_next'] <= 14)).astype(int)
df_visits['return_within_30d_true'] = ((df_visits['days_to_next'] >= 0) & (df_visits['days_to_next'] <= 30)).astype(int)

# Replace old labels with new ones
df_visits['return_within_30d'] = df_visits['return_within_30d_true']
df_visits['return_within_14d'] = df_visits['return_within_14d_true']

print(f"True 30-day return rate: {df_visits['return_within_30d'].mean():.2%}")
print(f"True 14-day return rate: {df_visits['return_within_14d'].mean():.2%}")

# Show distribution
print("\nReturn timing distribution (non-null only):")
gap_valid = df_visits['days_to_next'][df_visits['days_to_next'] >= 0]
print(gap_valid.describe())


In [ ]:
## 1.3 Recompute True Prior Utilization
# ====================================
# Fix: prior_visits_90d is likely flat/synthetic
# Rebuild from actual visit history

print("Step 2: Rebuilding prior utilization features...")

# Helper function for rolling counts
def rolling_count_within_days(x_dates, window):
    """Count visits within window before current date"""
    out = []
    dates_array = np.array(x_dates)
    for i, d in enumerate(dates_array):
        lo = pd.Timestamp(d) - pd.Timedelta(days=window)
        mask = (dates_array[:i] >= lo) & (dates_array[:i] < d)
        out.append(mask.sum())
    return pd.Series(out, index=x_dates.index)

# Calculate rolling visit counts
df_visits['visits_30d'] = df_visits.groupby('patient_id')['visit_date'].transform(
    lambda s: rolling_count_within_days(s, 30)
)
df_visits['visits_90d'] = df_visits.groupby('patient_id')['visit_date'].transform(
    lambda s: rolling_count_within_days(s, 90)
)
df_visits['visits_365d'] = df_visits.groupby('patient_id')['visit_date'].transform(
    lambda s: rolling_count_within_days(s, 365)
)

# Last visit gap
df_visits['prev_visit_date'] = df_visits.groupby('patient_id')['visit_date'].shift(1)
df_visits['days_since_last_visit'] = (df_visits['visit_date'] - df_visits['prev_visit_date']).dt.days
df_visits['days_since_last_visit'] = df_visits['days_since_last_visit'].fillna(9999)  # No prior visit

# Replace prior_visits_90d with real counts
df_visits['prior_visits_90d'] = df_visits['visits_90d']

print("Prior utilization stats:")
print(df_visits[['visits_30d', 'visits_90d', 'visits_365d', 'days_since_last_visit']].describe())

# Add utilization risk flags
df_visits['high_utilizer'] = (df_visits['visits_90d'] >= 3).astype(int)
df_visits['very_high_utilizer'] = (df_visits['visits_90d'] >= 5).astype(int)
df_visits['recent_visitor'] = (df_visits['days_since_last_visit'] <= 14).astype(int)


In [ ]:
# Load data
# Upload your CSV files to Colab
df_visits = pd.read_csv('mmc_opd_visits_2023_2025.csv', parse_dates=['visit_date'])

print(f"Total visits: {len(df_visits):,}")
print(f"Date range: {df_visits['visit_date'].min()} to {df_visits['visit_date'].max()}")

# Check target distribution
print("\nTarget distribution:")
print(df_visits['return_within_30d'].value_counts(normalize=True))
print(f"\nPositives: {df_visits['return_within_30d'].sum():,} ({df_visits['return_within_30d'].sum()/len(df_visits)*100:.1f}%)")


## 2. EDA & Data Validation


In [ ]:
# Remove leakage features (future information)
LEAKAGE_FEATURES = ['next_followup_date', 'followup_recommended', 'progress_status']

# Check for missing values
print("Missing values per column:")
missing = df_visits.isnull().sum()
print(missing[missing > 0])

# Check data types
print("\n" + "="*50)
print("Data Types and Ranges:")
print(df_visits.dtypes)

# Basic stats
print("\n" + "="*50)
print("Numeric Data Overview:")
print(df_visits.describe())

# Check target distribution
print("\n" + "="*50)
print("Target Distribution:")
print(df_visits['return_within_30d'].value_counts())
print(f"\nReturn rate: {df_visits['return_within_30d'].mean():.2%}")


In [ ]:
# Additional models: RandomForest, GradientBoosting, (optional) LightGBM
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score

# CRITICAL: Ensure v2 feature engineering is available
# If build_features doesn't exist, define it here (v2 with 46 features)
if 'build_features' not in globals():
    print("📝 Defining v2 build_features() function (46 features)...")
    
    def build_features(df, feature_version='v2'):
        """
        Enhanced feature builder with v2 engineering (46 features).
        Returns feature matrix X and feature names.
        """
        df = df.copy()
        
        # 1. Handle missing values first
        numeric_cols = ['age', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'bmi', 'prior_visits_90d', 'wait_time_minutes']
        for col in numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(df[col].median())
        
        # 2. Temporal features
        df['visit_date'] = pd.to_datetime(df['visit_date'])
        df['visit_day_of_week'] = df['visit_date'].dt.dayofweek
        df['visit_month'] = df['visit_date'].dt.month
        df['visit_week'] = df['visit_date'].dt.isocalendar().week
        
        # 3. Seasonality flags
        df['is_rainy_season'] = df['visit_month'].isin([6, 7, 8, 9, 10]).astype(int)
        df['is_cool_season'] = df['visit_month'].isin([11, 12, 1, 2]).astype(int)
        df['is_weekend'] = (df['visit_day_of_week'] >= 5).astype(int)
        
        # 4. Numeric feature engineering (fill NaN before categorizing)
        df['bp_level'] = (df['systolic_bp'] / df['diastolic_bp']).replace([np.inf, -np.inf], np.nan).fillna(2.0)
        
        # Age group with proper handling
        age_bins = pd.cut(df['age'], bins=[0, 18, 30, 50, 65, 100], labels=[0, 1, 2, 3, 4], include_lowest=True)
        df['age_group'] = age_bins.fillna(0).astype(int)
        
        # BMI category with proper handling
        bmi_bins = pd.cut(df['bmi'], bins=[0, 18.5, 25, 30, 100], labels=[0, 1, 2, 3], include_lowest=True)
        df['bmi_category'] = bmi_bins.fillna(1).astype(int)
        
        # Vital signs normalization
        df['bp_high'] = (df['systolic_bp'] > 140).astype(int)
        df['bp_low'] = (df['systolic_bp'] < 90).astype(int)
        df['heart_rate_abnormal'] = ((df['heart_rate'] < 60) | (df['heart_rate'] > 100)).astype(int)
        df['bmi_abnormal'] = ((df['bmi'] < 18.5) | (df['bmi'] > 30)).astype(int)
        
        # Visit frequency features
        df['frequent_visitor'] = (df['prior_visits_90d'] >= 3).astype(int)
        df['very_frequent_visitor'] = (df['prior_visits_90d'] >= 5).astype(int)
        
        # 4. Categorical encoding (keep top categories)
        categorical_features = [
            'diagnosis_group', 'service_type', 'sex', 'chronic_flag',
            'visit_session', 'provider_specialty'
        ]
        
        # One-hot encoding
        df_encoded = pd.get_dummies(df[categorical_features], prefix=categorical_features)
        
        # 5. Base numeric features (missing values already handled above)
        numeric_features = [
            'age', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'bmi',
            'prior_visits_90d', 'wait_time_minutes', 'bp_level'
        ]
        
        # Ensure pwd_flag exists (already handled if column exists)
        if 'pwd_flag' in df.columns:
            df['pwd_flag'] = df['pwd_flag'].fillna(0).astype(int)
            numeric_features.append('pwd_flag')
        
        # 6. Combine all features
        temporal_features = ['visit_day_of_week', 'visit_month', 'is_rainy_season', 
                             'is_cool_season', 'is_weekend', 'age_group', 'bmi_category']
        risk_features = ['bp_high', 'bp_low', 'heart_rate_abnormal', 'bmi_abnormal',
                         'frequent_visitor', 'very_frequent_visitor']
        
        feature_cols = (numeric_features + temporal_features + risk_features + 
                        list(df_encoded.columns))
        
        X = pd.concat([
            df[numeric_features + temporal_features + risk_features],
            df_encoded
        ], axis=1)
        
        # Ensure column order consistency
        X = X.reindex(columns=feature_cols, fill_value=0)
        
        return X, feature_cols
    
    print("✅ v2 build_features() function defined (46 features expected)")

# Now ensure train/val/test split exists
if 'X_train' not in globals() or 'y_train' not in globals():
    # If feature matrix X is missing, rebuild from df_visits
    if 'X' not in globals() or 'y' not in globals():
        # Verify we're using v2 feature builder
        print("🔧 Building features with v2 builder...")
        X, feature_names = build_features(df_visits)
        y = df_visits['return_within_30d'].values
        
        # Verify feature count
        n_features = len(feature_names)
        if n_features < 40:
            print(f"⚠️ WARNING: Only {n_features} features generated, expected 46 for v2!")
        else:
            print(f"✅ Generated {n_features} features (v2)")
    else:
        # X and y exist, verify feature count
        if len(feature_names) != 46:
            print(f"⚠️ WARNING: Feature count is {len(feature_names)}, expected 46 for v2!")
    
    train_mask = (df_visits['visit_date'] >= '2023-01-01') & (df_visits['visit_date'] < '2025-01-01')
    val_mask = (df_visits['visit_date'] >= '2025-01-01') & (df_visits['visit_date'] < '2025-07-01')
    test_mask = (df_visits['visit_date'] >= '2025-07-01')
    X_train, y_train = X[train_mask], y[train_mask]
    X_val, y_val = X[val_mask], y[val_mask]
    X_test, y_test = X[test_mask], y[test_mask]
    
    # Verify feature count matches
    print(f"✅ Training with {X_train.shape[1]} features (should be 46 for v2)")

HAS_LGBM = False
try:
    import lightgbm as lgb
    HAS_LGBM = True
except Exception:
    pass

models = {}

# Random Forest (good baseline for tabular)
rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=None,
    min_samples_split=4,
    min_samples_leaf=2,
    class_weight=None,  # Favor accuracy
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train, y_train)
proba_val_rf = rf.predict_proba(X_val)[:,1]
proba_test_rf = rf.predict_proba(X_test)[:,1]
models['RandomForest'] = (rf, proba_val_rf, proba_test_rf)

# Gradient Boosting (strong accuracy baseline)
gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.9,
    random_state=42
)
gb.fit(X_train, y_train)
proba_val_gb = gb.predict_proba(X_val)[:,1]
proba_test_gb = gb.predict_proba(X_test)[:,1]
models['GradientBoosting'] = (gb, proba_val_gb, proba_test_gb)

# LightGBM (if available)
if HAS_LGBM:
    lgbm = lgb.LGBMClassifier(
        n_estimators=600,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=0.0,
        random_state=42,
        verbose=-1  # Set verbose in constructor, not fit()
    )
    lgbm.fit(X_train, y_train,
             eval_set=[(X_val, y_val)],
             eval_metric='auc')
    proba_val_lgb = lgbm.predict_proba(X_val)[:,1]
    proba_test_lgb = lgbm.predict_proba(X_test)[:,1]
    models['LightGBM'] = (lgbm, proba_val_lgb, proba_test_lgb)

# Only using: GradientBoosting, RandomForest, LightGBM
# LR/XGB/CatBoost are skipped to avoid redundancy

print(f"✅ Trained models for comparison: {list(models.keys())}")
print(f"📊 Total models available: {len(models)}")
print("\nThese three models will be evaluated for accuracy in cell 11:")
print("  - RandomForest")
print("  - GradientBoosting") 
print("  - LightGBM (if available)")


In [ ]:
# Threshold search maximizing validation accuracy (and optional constraints)
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score

def find_threshold_max_accuracy(y_true, y_proba):
    best_t, best_acc = 0.5, 0.0
    for t in np.linspace(0.05, 0.95, 181):
        preds = (y_proba >= t).astype(int)
        acc = accuracy_score(y_true, preds)
        if acc > best_acc:
            best_acc, best_t = acc, t
    return best_t, best_acc

# Evaluate all models by validation accuracy
val_summary = []
for name, (m, proba_val, proba_test) in models.items():
    t_acc, acc_val = find_threshold_max_accuracy(y_val, proba_val)
    val_summary.append((name, acc_val, t_acc))

val_summary = sorted(val_summary, key=lambda x: x[1], reverse=True)
print("\nValidation Accuracy by model (maximized over threshold):")
for name, acc_val, t_acc in val_summary:
    print(f"  {name:<18} Acc@bestT={acc_val:.4f}  T={t_acc:.2f}")

best_name, best_acc, best_T = val_summary[0]
best_model, best_val_proba, best_test_proba = models[best_name]
print(f"\n✅ Best by validation accuracy: {best_name}  (Acc={best_acc:.4f} @ T={best_T:.2f})")

# Compute test accuracy at the chosen validation threshold
pred_test_best = (best_test_proba >= best_T).astype(int)
acc_test = accuracy_score(y_test, pred_test_best)
rec_test = recall_score(y_test, pred_test_best)
prec_test = precision_score(y_test, pred_test_best, zero_division=0)
print(f"Test Accuracy: {acc_test:.4f} | Test Recall: {rec_test:.4f} | Test Precision: {prec_test:.4f}")


In [ ]:
# Simple soft-voting ensemble (average of calibrated probabilities)
# Only ensemble the three selected models

# Choose models for ensembling (only the three we're using)
ensemble_members = []
for key in ['GradientBoosting', 'RandomForest', 'LightGBM']:
    if key in models:
        ensemble_members.append(key)

ensemble_created = False
if len(ensemble_members) >= 2:
    print(f"Ensembling: {ensemble_members}")
    val_stack = np.zeros_like(y_val, dtype=float)
    test_stack = np.zeros_like(y_test, dtype=float)
    for key in ensemble_members:
        _, pv, pt = models[key]
        val_stack += pv
        test_stack += pt
    val_stack /= len(ensemble_members)
    test_stack /= len(ensemble_members)
    ensemble_created = True

    # Evaluate ensemble by validation accuracy
    T_ens, acc_ens = find_threshold_max_accuracy(y_val, val_stack)
    pred_test_ens = (test_stack >= T_ens).astype(int)
    acc_test_ens = accuracy_score(y_test, pred_test_ens)
    rec_test_ens = recall_score(y_test, pred_test_ens)
    print(f"Ensemble Acc@ValBestT: {acc_ens:.4f} (T={T_ens:.2f}) | Test Acc: {acc_test_ens:.4f} | Test Recall: {rec_test_ens:.4f}")
else:
    print("Not enough models to ensemble.")
    val_stack = None
    test_stack = None
    T_ens = None
    acc_ens = 0.0


In [ ]:
# Report final model for accuracy-oriented policy (for manuscript compliance)
from sklearn.metrics import confusion_matrix, classification_report

# Ensure required variables exist
if 'best_name' not in globals() or 'best_acc' not in globals():
    print("ERROR: Please run the 'Threshold search' cell first to find the best model.")
    print("Models available:", list(models.keys()))
else:
    # Choose between best single model by accuracy and ensemble
    use_ensemble = False
    if ensemble_created and len(ensemble_members) >= 2 and val_stack is not None:
        # Prefer ensemble if it improves validation accuracy over best single model
        if acc_ens > best_acc + 1e-4:
            use_ensemble = True

    if use_ensemble:
        final_name = 'Ensemble(' + '+'.join(ensemble_members) + ')'
        final_threshold = T_ens
        final_proba_test = test_stack
    else:
        final_name = best_name
        final_threshold = best_T
        final_proba_test = best_test_proba

    print(f"\nFinal accuracy-oriented model: {final_name}")
    print(f"Threshold (validation-max accuracy): {final_threshold:.2f}")

    final_preds_test = (final_proba_test >= final_threshold).astype(int)
    acc = accuracy_score(y_test, final_preds_test)
    cm = confusion_matrix(y_test, final_preds_test)
    print(f"\nTest Accuracy: {acc:.4f}")
    print("Confusion Matrix (Test):")
    print(cm)
    print("\n" + classification_report(y_test, final_preds_test))


## 3. Feature Engineering


In [ ]:
def build_features(df, feature_version='v2'):
    """
    Enhanced feature builder with more engineering.
    Returns feature matrix X and feature names.
    """
    df = df.copy()
    
    # 1. Handle missing values first
    numeric_cols = ['age', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'bmi', 'prior_visits_90d', 'wait_time_minutes']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(df[col].median())
    
    # 2. Temporal features
    df['visit_date'] = pd.to_datetime(df['visit_date'])
    df['visit_day_of_week'] = df['visit_date'].dt.dayofweek
    df['visit_month'] = df['visit_date'].dt.month
    df['visit_week'] = df['visit_date'].dt.isocalendar().week
    
    # 3. Seasonality flags
    df['is_rainy_season'] = df['visit_month'].isin([6, 7, 8, 9, 10]).astype(int)
    df['is_cool_season'] = df['visit_month'].isin([11, 12, 1, 2]).astype(int)
    df['is_weekend'] = (df['visit_day_of_week'] >= 5).astype(int)
    
    # 4. Numeric feature engineering (fill NaN before categorizing)
    df['bp_level'] = (df['systolic_bp'] / df['diastolic_bp']).fillna(2.0)  # Default ratio
    
    # Age group with proper handling
    age_bins = pd.cut(df['age'], bins=[0, 18, 30, 50, 65, 100], labels=[0, 1, 2, 3, 4], include_lowest=True)
    # Fill NaN with 0 (existing category)
    df['age_group'] = age_bins.fillna(0).astype(int)
    
    # BMI category with proper handling
    bmi_bins = pd.cut(df['bmi'], bins=[0, 18.5, 25, 30, 100], labels=[0, 1, 2, 3], include_lowest=True)
    # Fill NaN with 1 (normal BMI category)
    df['bmi_category'] = bmi_bins.fillna(1).astype(int)
    
    # Vital signs normalization
    df['bp_high'] = (df['systolic_bp'] > 140).astype(int)
    df['bp_low'] = (df['systolic_bp'] < 90).astype(int)
    df['heart_rate_abnormal'] = ((df['heart_rate'] < 60) | (df['heart_rate'] > 100)).astype(int)
    df['bmi_abnormal'] = ((df['bmi'] < 18.5) | (df['bmi'] > 30)).astype(int)
    
    # Visit frequency features
    df['frequent_visitor'] = (df['prior_visits_90d'] >= 3).astype(int)
    df['very_frequent_visitor'] = (df['prior_visits_90d'] >= 5).astype(int)
    
    # 4. Categorical encoding (keep top categories)
    categorical_features = [
        'diagnosis_group', 'service_type', 'sex', 'chronic_flag',
        'visit_session', 'provider_specialty'
    ]
    
    # One-hot encoding
    df_encoded = pd.get_dummies(df[categorical_features], prefix=categorical_features)
    
    # 5. Base numeric features (missing values already handled above)
    numeric_features = [
        'age', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'bmi',
        'prior_visits_90d', 'wait_time_minutes', 'bp_level'
    ]
    
    # Ensure pwd_flag exists (already handled if column exists)
    if 'pwd_flag' in df.columns:
        df['pwd_flag'] = df['pwd_flag'].fillna(0).astype(int)
        numeric_features.append('pwd_flag')
    
    # 6. Combine all features
    temporal_features = ['visit_day_of_week', 'visit_month', 'is_rainy_season', 
                         'is_cool_season', 'is_weekend', 'age_group', 'bmi_category']
    risk_features = ['bp_high', 'bp_low', 'heart_rate_abnormal', 'bmi_abnormal',
                     'frequent_visitor', 'very_frequent_visitor']
    
    feature_cols = (numeric_features + temporal_features + risk_features + 
                    list(df_encoded.columns))
    
    X = pd.concat([
        df[numeric_features + temporal_features + risk_features],
        df_encoded
    ], axis=1)
    
    # Ensure column order consistency
    X = X.reindex(columns=feature_cols, fill_value=0)
    
    return X, feature_cols

# Build features
X, feature_names = build_features(df_visits)
y = df_visits['return_within_30d'].values

print(f"Feature matrix shape: {X.shape}")
print(f"Number of features: {len(feature_names)}")
print(f"\nTarget distribution:")
print(f"Positives: {y.sum():,} ({y.sum()/len(y)*100:.2f}%)")
print(f"Negatives: {(y==0).sum():,} ({(y==0).sum()/len(y)*100:.2f}%)")


## 4. Time-Aware Train/Test Split


In [ ]:
# Split by date ranges
train_mask = (df_visits['visit_date'] >= '2023-01-01') & (df_visits['visit_date'] < '2025-01-01')
val_mask = (df_visits['visit_date'] >= '2025-01-01') & (df_visits['visit_date'] < '2025-07-01')
test_mask = (df_visits['visit_date'] >= '2025-07-01')

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f"Train: {len(X_train):,} samples")
print(f"Val:   {len(X_val):,} samples")
print(f"Test:  {len(X_test):,} samples")
print(f"\nTrain positives: {y_train.sum():,} ({y_train.sum()/len(y_train)*100:.1f}%)")
print(f"Val positives: {y_val.sum():,} ({y_val.sum()/len(y_val)*100:.1f}%)")
print(f"Test positives: {y_test.sum():,} ({y_test.sum()/len(y_test)*100:.1f}%)")


## 5. Model Training

**Note:** Model training happens in **Cell 10** below.  
This section (cells 19-20) contained old LR/XGB training which is now deprecated.  
Proceed directly to **Cell 10** for the three-model comparison.


In [ ]:
# ============================================================================
# DEPRECATED: CatBoost training - SKIPPED
# ⚠️ Using only GradientBoosting, RandomForest, and LightGBM (see cell 10)
# ============================================================================
print("⚠️ CatBoost training skipped")
print("✅ Using GradientBoosting, RandomForest, and LightGBM from cell 10 instead")
print("\n📋 Proceed to cell 10 to train the three selected models")


In [ ]:
# Model Comparison
print("\n" + "="*60)
print("MODEL PERFORMANCE COMPARISON")
print("="*60)

if HAS_XGBOOST:
    print("\nLogistic Regression:")
    print(f"  Val AUROC:   {val_auc:.4f} | Test AUROC: {test_auc:.4f}")
    print(f"  Val PR-AUC:  {val_pr_auc:.4f} | Test PR-AUC: {test_pr_auc:.4f}")
    
    print("\nXGBoost:")
    print(f"  Val AUROC:   {val_auc_xgb:.4f} | Test AUROC: {test_auc_xgb:.4f}")
    print(f"  Val PR-AUC:  {val_pr_auc_xgb:.4f} | Test PR-AUC: {test_pr_auc_xgb:.4f}")
    
    # Choose best model
    if test_auc_xgb > test_auc:
        chosen_model = xgb_model
        chosen_proba = y_test_pred_proba_xgb
        chosen_val_proba = y_val_pred_proba_xgb
        model_name = "XGBoost"
        use_scaled = False
        print(f"\n✅ Best Model: XGBoost (AUROC: {test_auc_xgb:.4f})")
    else:
        chosen_model = lr_model
        chosen_proba = y_test_pred_proba
        chosen_val_proba = y_val_pred_proba
        model_name = "LogisticRegression"
        use_scaled = True
        print(f"\n✅ Best Model: Logistic Regression (AUROC: {test_auc:.4f})")
else:
    chosen_model = lr_model
    chosen_proba = y_test_pred_proba
    chosen_val_proba = y_val_pred_proba
    model_name = "LogisticRegression"
    use_scaled = True
    print(f"\n✅ Using: Logistic Regression (AUROC: {test_auc:.4f})")

print("="*60)


## 7. Explainability with SHAP


In [ ]:
# Feature Importance Analysis (for final selected model)
# Use the best model from accuracy selection (cell 13)

if 'best_model' in globals() and 'best_name' in globals():
    print(f"\n📊 Feature Importance Analysis for: {best_name}")
    print("="*60)
    
    # For tree-based models (RandomForest, GradientBoosting, LightGBM)
    if best_name in ['RandomForest', 'GradientBoosting', 'LightGBM']:
        try:
            importances = best_model.feature_importances_
            feature_importance = pd.DataFrame({
                'feature': feature_names[:len(importances)],
                'importance': importances
            }).sort_values('importance', ascending=False)
            
            print("\nTop 15 Feature Importance:")
            print(feature_importance.head(15).to_string(index=False))
            
            # Optional: SHAP for tree models if available
            if HAS_SHAP and best_name in ['RandomForest', 'GradientBoosting', 'LightGBM']:
                try:
                    explainer = shap.TreeExplainer(best_model)
                    shap_values = explainer.shap_values(X_val[:100])  # Sample for speed
                    
                    plt.figure(figsize=(10, 8))
                    shap.summary_plot(shap_values, X_val[:100], show=False)
                    plt.title(f"SHAP Feature Importance - {best_name}")
                    plt.tight_layout()
                    plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
                    plt.close()
                    
                    print("\n✅ SHAP summary plot saved as 'shap_summary.png'")
                except Exception as e:
                    print(f"SHAP analysis skipped: {e}")
        except Exception as e:
            print(f"Feature importance analysis failed: {e}")
    else:
        print(f"Feature importance for {best_name} model")
else:
    print("⚠️ Run cell 11 first to select the best model")
    print("Available models:", list(models.keys()) if 'models' in globals() else "None")


## 8. Export Model Artifacts


In [ ]:
import pickle
import json

# Use the accuracy-optimized model from cell 13 (or fallback to best single model)
if 'final_name' in globals() and 'final_threshold' in globals() and 'final_proba_test' in globals():
    # Use accuracy-optimized model
    save_threshold = final_threshold
    save_model_name = final_name
    save_test_acc = acc
    
    # If ensemble was used, save the best single model instead (can't pickle ensemble directly)
    if use_ensemble and 'best_model' in globals():
        save_model = best_model
        print(f"Using accuracy-optimized model: {save_model_name} (saving best single model: {best_name})")
    elif 'best_model' in globals():
        save_model = best_model
        print(f"Using accuracy-optimized model: {save_model_name}")
    else:
        save_model = None
        print("WARNING: Model object not found, only saving metrics")
else:
    # Fallback: use best single model from threshold search
    if 'best_model' in globals() and 'best_T' in globals():
        save_model = best_model
        save_threshold = best_T
        save_model_name = best_name
        save_test_acc = accuracy_score(y_test, (best_test_proba >= best_T).astype(int))
        print(f"Using best single model: {save_model_name}")
    else:
        print("ERROR: Run cells 10-13 first to train models and select best one!")
        save_model = None

if save_model is not None:
    # CRITICAL: Verify model and feature count match before exporting
    try:
        # Check model's expected feature count
        if hasattr(save_model, 'booster_'):
            model_n_features = save_model.booster_.num_feature()
        elif hasattr(save_model, '_Booster'):
            model_n_features = save_model._Booster.num_feature()
        elif hasattr(save_model, 'n_features_in_'):
            model_n_features = save_model.n_features_in_
        else:
            # For tree-based models, try to infer from training data shape
            model_n_features = X_train.shape[1]
        
        # Check feature list count
        if 'feature_names' in globals():
            export_n_features = len(feature_names)
        else:
            raise RuntimeError("ERROR: feature_names not found! Run Cell 15 (Feature Engineering) first.")
        
        # Verify match
        if model_n_features != export_n_features:
            raise RuntimeError(
                f"CRITICAL MISMATCH DETECTED!\n"
                f"Model was trained with {model_n_features} features\n"
                f"But feature_list.json will export {export_n_features} features\n"
                f"This will cause deployment errors!\n\n"
                f"SOLUTION: Retrain the model by running cells in order:\n"
                f"  1. Cell 15 (Feature Engineering) - defines v2 build_features()\n"
                f"  2. Cell 17 (Train/Test Split)\n"
                f"  3. Cell 10 (Model Training)\n"
                f"  4. Cell 13 (Model Selection)\n"
                f"  5. This cell (Export)\n"
            )
        
        if export_n_features != 46:
            print(f"⚠️ WARNING: Exporting {export_n_features} features, expected 46 for v2")
            print("   Model may not match batch_scorer.py expectations")
        
        print(f"✅ Verification passed: Model and features both use {model_n_features} features (v2)")
        
    except Exception as e:
        print(f"⚠️ WARNING: Could not verify feature count match: {e}")
        print("   Proceeding with export, but verify manually before deployment!")
    
    # Save model
    with open('model.pkl', 'wb') as f:
        pickle.dump(save_model, f)
    print("Saved: model.pkl")

    # Save feature list
    if 'feature_names' in globals():
        feature_config = {
            'feature_names': feature_names,
            'feature_version': 'v2',
            'n_features': len(feature_names)
        }
        with open('feature_list.json', 'w') as f:
            json.dump(feature_config, f, indent=2)
        print("Saved: feature_list.json")
        print(f"   Exporting {len(feature_names)} features (expected: 46 for v2)")

    # Save threshold and metrics
    metrics = {
        'model_version': 'v0.2.0',
        'threshold': float(save_threshold),
        'test_accuracy': float(save_test_acc),
        'model_type': save_model_name,
        'optimization': 'accuracy_maximized',
        'trained_on_start': '2023-01-01',
        'trained_on_end': '2024-12-31',
        'n_train': len(X_train),
        'n_val': len(X_val),
        'n_test': len(X_test)
    }
    with open('threshold.json', 'w') as f:
        json.dump(metrics, f, indent=2)
    print("Saved: threshold.json")

    print("\nModel artifacts ready for deployment!")


In [ ]:
# Export test set predictions for dashboard smoke test
# Use accuracy-optimized predictions
if 'final_proba_test' in globals() and 'final_threshold' in globals() and 'final_preds_test' in globals():
    score_array = final_proba_test
    threshold_val = final_threshold
    preds_array = final_preds_test
    print("Using accuracy-optimized predictions")
elif 'best_test_proba' in globals() and 'best_T' in globals():
    score_array = best_test_proba
    threshold_val = best_T
    preds_array = (best_test_proba >= best_T).astype(int)
    print("Using best single model predictions")
else:
    print("ERROR: Run model training and selection cells first!")
    score_array = None

if score_array is not None:
    # Ensure we have the test_mask
    if 'test_mask' not in globals():
        test_mask = (df_visits['visit_date'] >= '2025-07-01')
    
    df_test_predictions = pd.DataFrame({
        'visit_id': df_visits.loc[test_mask, 'visit_id'].values if len(df_visits.loc[test_mask]) > 0 
                    else [f'test_{i}' for i in range(len(score_array))],
        'patient_id': df_visits.loc[test_mask, 'patient_id'].values if len(df_visits.loc[test_mask]) > 0 
                     else [f'patient_{i}' for i in range(len(score_array))],
        'score': score_array,
        'risk_tier': pd.cut(score_array, bins=[0, threshold_val, 1], labels=['Low', 'High']),
        'actual_return': y_test,
        'predicted_return': preds_array
    })

    df_test_predictions.to_csv('test_predictions.csv', index=False)
    print(f"Saved: test_predictions.csv ({len(df_test_predictions)} rows)")
